In [1]:
from pyspark.sql import SparkSession

In [46]:
from pyspark.sql.functions import col ,udf , StringType

In [3]:
spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 04:29:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 04:29:47 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)


In [6]:
print("Before partitioning",df.rdd.getNumPartitions())

Before partitioning 2


In [7]:
df_repartitioned=df.repartition(10)

In [9]:
print("After Partition",df_repartitioned.rdd.getNumPartitions())

[Stage 0:=============================>                             (1 + 1) / 2]

After Partition 10


In [10]:
df_coalesced=df_repartitioned.coalesce(2)

In [11]:
print("After Coalesce",df_coalesced.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

After Coalesce 2


In [12]:
df_repartitioned20=df.repartition(20)

In [13]:
df_repartitioned20.write.mode("overwrite").csv("output/after-partition-data",header=True)

In [14]:
df.write.mode("overwrite").csv("output/original-partition-data",header=True)

In [15]:
df_coalesced5=df_repartitioned.coalesce(5)

In [16]:
df_coalesced5.write.mode("overwrite").csv("output/coalesce-partition-data",header=True)

In [32]:
import time

In [33]:
optimized_df=df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [34]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [35]:
start_time=time.time()
count_uncached=optimized_df.count()
end_time=time.time()
print(f"1. Optimized execution | count :{count_uncached} | Time : {round(end_time-start_time,4)} seconds")

1. Optimized execution | count :2495000 | Time : 0.1857 seconds


In [36]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [37]:
start_time=time.time()
count_cached=optimized_df.count()
end_time=time.time()
print(f"2. Optimized execution | count :{count_cached} | Time : {round(end_time-start_time,4)} seconds")

[Stage 25:=============================>                            (1 + 1) / 2]

2. Optimized execution | count :2495000 | Time : 1.2074 seconds


In [38]:
start_time=time.time()
count_cached=optimized_df.count()
end_time=time.time()
print(f"3. Optimized execution | count :{count_cached} | Time : {round(end_time-start_time,4)} seconds")

3. Optimized execution | count :2495000 | Time : 0.1558 seconds


In [39]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [40]:
def can_drive(age):
    if age>=18:
        return "Can Drive"
    elif age>=13 :
        return "Drive with Learning Licence"
    else:
        return "Cannot Drive"

In [41]:
data=[("A",25),("B",14),("C",11),("D",20),("E",8)]
columns=["Name","Age"]
d1=spark.createDataFrame(data,columns)

In [42]:
d1.show()

[Stage 32:>                                                         (0 + 1) / 1]

+----+---+
|Name|Age|
+----+---+
|   A| 25|
|   B| 14|
|   C| 11|
|   D| 20|
|   E|  8|
+----+---+



In [47]:
drive_udf=udf(can_drive,StringType())

In [48]:
df_method=d1.withColumn("drive",drive_udf(col("Age")))
print("method-1 ")
df_method.show()

method-1 


+----+---+--------------------+
|Name|Age|               drive|
+----+---+--------------------+
|   A| 25|           Can Drive|
|   B| 14|Drive with Learni...|
|   C| 11|        Cannot Drive|
|   D| 20|           Can Drive|
|   E|  8|        Cannot Drive|
+----+---+--------------------+



In [49]:
#for sql
spark.udf.register("sql_drive",can_drive,StringType())
d1.createOrReplaceTempView("people")

In [50]:
sql_df=spark.sql('''
select Name, Age , sql_drive(Age) as DriveAPPlicable from people ''')
sql_df.show()

+----+---+--------------------+
|Name|Age|     DriveAPPlicable|
+----+---+--------------------+
|   A| 25|           Can Drive|
|   B| 14|Drive with Learni...|
|   C| 11|        Cannot Drive|
|   D| 20|           Can Drive|
|   E|  8|        Cannot Drive|
+----+---+--------------------+

